# Metering for AI Units

You have AI functionality in your application that is commercialised using AI units.

Your application thus needs to report a business metrics that reflects the business value of that AI functionality and covers the cost caused by GenAI / LLM services, e.g. processed documents. 

The consumption of AI units per customer tenant and the underlying business metrics should be reflected in billing and for customer transparency in SAP4ME.

For further information see: [Metering for AI Units](https://workzone.one.int.sap/site#workzone-home&/wiki/show/PjZrYMoGizdSSi1iJ68Plq)

## Metering using the Generative AI Hub

The generative AI hub is a central piece of the GenAI architecture. 

If it is used directly by the application and the business metrics definition follows certain supported patterns, the generative AI hub can be used to report business metrics to Unified Metering on behalf of the application. 

Each LLM access request to the generative AI hub needs to supply additional information in the request as documented here: [Documentation for providing additional metering relevant metadata in an LLM request](https://help.sap.com/doc/4317866f8cb44a089995b8444dc6c707/INTERNAL/en-US/553250b6ec764a05be43a7cd8cba0526.pdf) (page 10).

The headers required for metering are:
- `X-USECASE-ID`: "identifier"
- `X-BUSINESS-CONTEXT`: "value"
- `X-LOCALTENANT-ID`: "unique identifier" 
- `X-PRODUCT-TYPE`: "value"

## Enabling Metering using the SDK

**There are two ways to set headers for metering:**

1. **Instance-level headers**: Applied to all requests made by the proxy client instance.

2. **Request-level headers**: Applied only to requests within a context manager block.

### Common Setup

Define your metering headers once and reuse them across different clients:

In [ ]:
from gen_ai_hub.proxy import get_proxy_client
from gen_ai_hub.proxy.gen_ai_hub_proxy import temporary_headers_addition

METERING_HEADERS = {
    'X-USECASE-ID': 'my-usecase',
    'X-BUSINESS-CONTEXT': 'my-context',
    'X-LOCALTENANT-ID': 'tenant-123',
    'X-PRODUCT-TYPE': 'my-product'
}

# Create a proxy client with instance-level headers
proxy_client = get_proxy_client('gen-ai-hub')
proxy_client.set_headers_addition(headers=METERING_HEADERS)

---
## Native LLM Clients

### Instance-level headers

In [ ]:
from gen_ai_hub.proxy.native.openai import OpenAI

# All requests from this client include metering headers
client = OpenAI(proxy_client=proxy_client)
response = client.chat.completions.create(
    model='gpt-4o',
    messages=[{'role': 'user', 'content': 'Hello!'}]
)
print(response.choices[0].message.content)

### Request-level headers

In [ ]:
from gen_ai_hub.proxy.native.openai import OpenAI

client = OpenAI()

# Only this request includes metering headers
with temporary_headers_addition(headers=METERING_HEADERS):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': 'Hello!'}]
    )
    print(response.choices[0].message.content)

---
## Orchestration Service

### Instance-level headers

In [ ]:
from gen_ai_hub.orchestration_v2 import (OrchestrationService, OrchestrationConfig, ModuleConfig, Template,
                                         PromptTemplatingModuleConfig, LLMModelDetails, UserMessage)

config = OrchestrationConfig(
    modules=ModuleConfig(
        prompt_templating=PromptTemplatingModuleConfig(
            prompt=Template(template=[UserMessage(content='{{?input}}')]),
            model=LLMModelDetails(name='gpt-4o')
        )
    )
)

# All requests from this service include metering headers
service = OrchestrationService(proxy_client=proxy_client, config=config)
response = service.run(placeholder_values={'input': 'Hello!'})
print(response.final_result.choices[0].message.content)

### Request-level headers

In [ ]:
from gen_ai_hub.orchestration_v2 import OrchestrationService

service = OrchestrationService(config=config)

# Only this request includes metering headers
with temporary_headers_addition(headers=METERING_HEADERS):
    response = service.run(placeholder_values={'input': 'Hello!'})
    print(response.final_result.choices[0].message.content)

---
## Prompt Registry

### Instance-level headers

In [ ]:
from gen_ai_hub.prompt_registry import PromptTemplateClient

# All requests from this client include metering headers
client = PromptTemplateClient(proxy_client=proxy_client)
templates = client.get_prompt_templates(scenario='my-scenario')
print(f"Found {templates.count} templates")

### Request-level headers

In [ ]:
from gen_ai_hub.prompt_registry import PromptTemplateClient

client = PromptTemplateClient()

# Only this request includes metering headers
with temporary_headers_addition(headers=METERING_HEADERS):
    templates = client.get_prompt_templates(scenario='my-scenario')
    print(f"Found {templates.count} templates")

---
## Document Grounding Clients

The document grounding clients (`PipelineAPIClient`, `RetrievalAPIClient`, `VectorAPIClient`) support the same header injection pattern.

### Instance-level headers

In [ ]:
from gen_ai_hub.document_grounding import PipelineAPIClient, RetrievalAPIClient, VectorAPIClient

# All requests from these clients include metering headers
pipeline_client = PipelineAPIClient(proxy_client=proxy_client)
retrieval_client = RetrievalAPIClient(proxy_client=proxy_client)
vector_client = VectorAPIClient(proxy_client=proxy_client)

pipelines = pipeline_client.get_pipelines()
repositories = retrieval_client.get_data_repositories()
collections = vector_client.get_collections()

print(f"Found {pipelines.count} pipelines")
print(f"Found {repositories.count} repositories")
print(f"Found {collections.count} collections")

### Request-level headers

In [ ]:
from gen_ai_hub.document_grounding import RetrievalAPIClient

client = RetrievalAPIClient()

# Only this request includes metering headers
with temporary_headers_addition(headers=METERING_HEADERS):
    repositories = client.get_data_repositories()
    print(f"Found {repositories.count} repositories")

---
## Combining Instance and Request-level Headers

Request-level headers are merged with instance-level headers. If the same header is set at both levels, the request-level value takes precedence.

In [ ]:
from gen_ai_hub.proxy.native.openai import OpenAI

# Set instance-level headers
proxy_client.set_headers_addition({
    'X-USECASE-ID': 'default-usecase',
    'X-LOCALTENANT-ID': 'tenant-123'
})

client = OpenAI(proxy_client=proxy_client)

# Override X-USECASE-ID for this specific request
with temporary_headers_addition({'X-USECASE-ID': 'special-usecase'}):
    # Request will have:
    # X-USECASE-ID: 'special-usecase' (from request-level)
    # X-LOCALTENANT-ID: 'tenant-123' (from instance-level)
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': 'Hello!'}]
    )